In [1]:
import os
import sys
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# CHANGE WORKING DIRECTORY TO ROOT
current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..")
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass
else:
    os.chdir("../..")
from src._utils._generate_dataset import main_generate_dataset

#############################################
# LOAD MODEL
#############################################

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda",
    attn_implementation="flash_attention_2",
    quantization_config=quantization_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.generation_config.pad_token_id = tokenizer.pad_token_id

# get true labels
df_real = pd.read_csv("real_data/train/agnewstrainAll.csv").rename(
    columns={"2": "text", "3": "label"}
)
correct_labels = df_real["label"].unique().tolist()

In [ ]:
#############################################
# GENERATE BASELINE AGNEWS DATASET
#############################################

prompt = f"""\
You are an expert in journalism and NLP specializing in news classification. \
Your task is to generate 10 high-quality short documents, that talks about the following four News categories:  
- **Business**
- **Sci/Tech**
- **Sports**
- **World**

### **Output Format (JSON)**  
Return only a valid JSON list of 10 items in the following structure:

```json
[
    {{"text": <text>, "label": <label>}},
    ...
]
```
"""

config = {
    "dataset": "agnews",
    "model": model,
    "tokenizer": tokenizer,
    "generation_method": "baseline",
    "prompt": prompt,
    "system_prompt": None,
    "num_examples": 500,
    "max_new_tokens": 4096,
    "seed": 42,
    "json_output_file": "synthetic_data/datasets/syn_agnews_baseline_500.json",
    "log_file": "src/agnews/generate_dataset_agnews_log.json",
    "correct_labels": correct_labels,
    "correct_fields": ["text", "label"],
}
main_generate_dataset(config)

In [ ]:
#############################################
# GENERATE TARGETED + TAGS AGNEWS DATASET
#############################################

prompt = f"""\
You are an expert in journalism and NLP specializing in news classification. \
Your task is to generate 10 high-quality short documents, that talks about the following four News categories (labels):  
- **Business**
- **Sci/Tech**
- **Sports**
- **World**

For each example, also list the key phenomena it covers.

### **Follow these topics:**
- **Business**  
  - Markets  
  - Economy  
  - Companies  
  - Startups  
  - Regulations  

- **Sci/Tech**  
  - AI  
  - Space  
  - Cybersecurity  
  - Biotech  
  - Climate  

- **Sports**  
  - Events  
  - Records  
  - Highlights  
  - Scandals  
  - Olympics  

- **World**  
  - Politics  
  - Conflicts  
  - Disasters  
  - Human Rights  
  - Trade

### **Output Format (JSON)**
The labels must be one of the specified categories, which are: Business, Sci/Tech, Sports, World.
Return only a valid JSON list of 10 elements in the following structure:

```json
[
    {{"text": <text of the document>, "label": <corresponding label>, "phenomena": ["<phenomenon1>", "<phenomenon2>", ...]}},
    ...
]
```
"""

config = {
    "dataset": "agnews",
    "model": model,
    "tokenizer": tokenizer,
    "generation_method": "targeted + linguistic tags",
    "prompt": prompt,
    "system_prompt": None,
    "num_examples": 500,
    "max_new_tokens": 4096,
    "seed": 42,
    "json_output_file": "synthetic_data/datasets/syn_agnews_targeted+tags_500.json",
    "log_file": "src/agnews/generate_dataset_agnews_log.json",
    "correct_labels": correct_labels,
    "correct_fields": ["text", "label", "phenomena"],
}
main_generate_dataset(config)

In [3]:
#############################################
# GENERATE TARGETED + TAGS AGNEWS DATASET (generate other 500)
#############################################

config['seed'] = config['seed']*8
config['json_output_file'] = "synthetic_data/datasets/syn_agnews_targeted+tags_500_2.json"

main_generate_dataset(config)


🚀 Starting Synthetic Dataset Generation
📊 Dataset              : agnews
📚 Generation method    : targeted + linguistic tags
🤖 Model                : deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
🔢 Examples to Generate : 500
💾 Output File          : synthetic_data/datasets/syn_agnews_targeted+tags_500_2.json
🕹️  Max New Tokens       : 4096
🎯 Seed                 : 336



Generating Examples:  28%|██▊       | 141/500 [04:25<10:19,  1.73s/ex, examples=141/500, run=11]

❌ Failed to parse generation 11: Expecting value: line 1 column 1 (char 0)


Generating Examples:  28%|██▊       | 141/500 [04:48<10:19,  1.73s/ex, examples=141/500, run=12]

❌ Failed to parse generation 12: Expecting ',' delimiter: line 4 column 9 (char 271)


...

❌ Failed to parse generation 43: Expecting value: line 1 column 1 (char 0)


Generating Examples:  93%|█████████▎| 465/500 [21:43<01:29,  2.56s/ex, examples=465/500, run=48]

❌ Failed to parse generation 48: Expecting value: line 1 column 1 (char 0)


Generating Examples: 100%|██████████| 500/500 [23:00<00:00,  2.76s/ex, examples=500/500, run=52]

📝 Log saved successfully to: src/agnews/generate_dataset_agnews_log.json
💾 Dataset with metadata saved to: synthetic_data/datasets/syn_agnews_targeted+tags_500_2.json
